In [1]:
import pandas as pd
import numpy as np
from math import log2
from collections import Counter
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.model_selection import cross_val_score, StratifiedKFold

In [ ]:
class Rule:
    def __init__(self, conditions, decision):
        self.conditions = conditions 
        self.decision = decision

    @property
    def length(self):
        return len(self.conditions)

    def get_support(self, T, decision_col):
        filtered = T
        for (attr, val) in self.conditions:
            filtered = filtered[filtered[attr] == val]
        if len(filtered) == 0:
            return 0
        return (filtered[decision_col] == self.decision).sum()

    def is_true(self, T, decision_col):
        filtered = T
        for (attr, val) in self.conditions:
            filtered = filtered[filtered[attr] == val]
        if len(filtered) == 0:
            return True
        return (filtered[decision_col] == self.decision).all()

    def is_realizable(self, row):
        for (attr, val) in self.conditions:
            if row[attr] != val:
                return False
        return True

    def __repr__(self):
        if not self.conditions:
            return f"-> d={self.decision}"
        conds = " AND ".join(f"{a}={v}" for a, v in self.conditions)
        return f"IF {conds} THEN d={self.decision}"

    def __eq__(self, other):
        if not isinstance(other, Rule):
            return False
        return (set(self.conditions) == set(other.conditions)
                and self.decision == other.decision)

    def __hash__(self):
        return hash((frozenset(self.conditions), self.decision))

In [3]:
def N(T, decision_col, a=None):
    if a is None:
        return len(T)
    return (T[decision_col] == a).sum()

def M(T, decision_col, a):
    return N(T, decision_col) - N(T, decision_col, a)

def mcd(T, decision_col):
    counts = T[decision_col].value_counts()
    return counts.index[0]

def E(T, decision_col):
    feature_cols = [c for c in T.columns if c != decision_col]
    return [c for c in feature_cols if T[c].nunique() > 1]

In [4]:
def induce_rule_for_row(T, row, decision_col, heuristic='RM'):
    a = row[decision_col]  
    feature_cols = [c for c in T.columns if c != decision_col]

    Q = []           
    T_j = T.copy()   

    while True:
        if len(T_j) == 0 or T_j[decision_col].nunique() <= 1:
            break

        available_attrs = [fi for fi in feature_cols
                          if fi not in Q and T_j[fi].nunique() > 1]

        if not available_attrs:
            remaining = [fi for fi in feature_cols if fi not in Q]
            if not remaining:
                break
            available_attrs = remaining

        best_attr = None
        best_score = None

        for fi in available_attrs:
            bi = row[fi]  

            T_j1 = T_j[T_j[fi] == bi]

            if len(T_j1) == 0:
                continue

            N_Tj1 = len(T_j1)
            N_Tj1_a = N(T_j1, decision_col, a)
            M_Tj1_a = N_Tj1 - N_Tj1_a

            N_Tj_a = N(T_j, decision_col, a)
            M_Tj_a = M(T_j, decision_col, a)
            alpha = N_Tj_a - N_Tj1_a
            beta = M_Tj_a - M_Tj1_a

            if heuristic == 'RM':
                score = M_Tj1_a / N_Tj1 if N_Tj1 > 0 else float('inf')
                if best_score is None or score < best_score:
                    best_score = score
                    best_attr = fi
                elif score == best_score and best_attr is not None:
                    if feature_cols.index(fi) < feature_cols.index(best_attr):
                        best_attr = fi

            elif heuristic == 'Poly':
                score = beta / (alpha + 1) if (alpha + 1) != 0 else 0
                if best_score is None or score > best_score:
                    best_score = score
                    best_attr = fi
                elif score == best_score and best_attr is not None:
                    if feature_cols.index(fi) < feature_cols.index(best_attr):
                        best_attr = fi


        if best_attr is None:
            break

        Q.append(best_attr)
        bi = row[best_attr]
        T_j = T_j[T_j[best_attr] == bi]

    conditions = [(fi, row[fi]) for fi in Q]
    return Rule(conditions, a)

In [5]:
def induce_rules_for_table(T, decision_col, heuristic='RM'):
    rules = {}
    for idx, row in T.iterrows():
        rule = induce_rule_for_row(T, row, decision_col, heuristic)
        rules[idx] = rule
    return rules

In [6]:
def global_optimization_length(rules_rm, rules_poly):
    result = {}
    all_indices = set(rules_rm.keys()) | set(rules_poly.keys())

    for idx in all_indices:
        rm = rules_rm.get(idx)
        poly = rules_poly.get(idx)

        if rm is None:
            result[idx] = [poly]
        elif poly is None:
            result[idx] = [rm]
        else:
            result[idx] =  [poly] if poly.length <= rm.length else [rm]

    return result


def global_optimization_support(rules_rm, rules_poly, T, decision_col):
    result = {}
    all_indices = set(rules_rm.keys()) | set(rules_poly.keys())

    for idx in all_indices:
        rm = rules_rm.get(idx)
        poly = rules_poly.get(idx)

        if rm is None:
            result[idx] = [poly]
        elif poly is None:
            result[idx] = [rm]
        else:
            rm_supp = rm.get_support(T, decision_col)
            poly_supp = poly.get_support(T, decision_col)
            result[idx] = [poly] if poly_supp >= rm_supp else [rm]

    return result

In [7]:
def get_unique_rules(rules_dict):
    unique = set()
    for idx, rule_or_list in rules_dict.items():
        if isinstance(rule_or_list, list):
            for r in rule_or_list:
                unique.add(r)
        else:
            unique.add(rule_or_list)
    return unique


def compute_statistics(rules_dict, T, decision_col, is_global=False):
    unique_rules = get_unique_rules(rules_dict)
    if not unique_rules:
        return {}

    lengths = [r.length for r in unique_rules]
    supports = [r.get_support(T, decision_col) for r in unique_rules]

    return {
        'num_unique_rules': len(unique_rules),
        'min_length': min(lengths),
        'avg_length': round(np.mean(lengths), 2),
        'max_length': max(lengths),
        'min_support': min(supports),
        'avg_support': round(np.mean(supports), 2),
        'max_support': max(supports),
    }


def build_statistics_dataframe(stats_dict):
    rows = []
    for name, stats in stats_dict.items():
        rows.append({
            'Metoda': name,
            'Liczba unikalnych regul': stats['num_unique_rules'],
            'Min dlugosc': stats['min_length'],
            'Avg dlugosc': stats['avg_length'],
            'Max dlugosc': stats['max_length'],
            'Min wsparcie': stats['min_support'],
            'Avg wsparcie': stats['avg_support'],
            'Max wsparcie': stats['max_support'],
        })
    return pd.DataFrame(rows)

In [8]:
class RuleBasedClassifier(BaseEstimator, ClassifierMixin):
    def __init__(self, heuristic='RM', decision_col='d'):
        self.heuristic = heuristic
        self.decision_col = decision_col
        self.rules_ = {}
        self.default_class_ = None

    def fit(self, X, y):
        T = X.copy()
        T[self.decision_col] = y.values
        self.default_class_ = y.mode()[0]
        self.rules_ = induce_rules_for_table(T, self.decision_col, self.heuristic)
        return self

    def predict(self, X):
        return np.array([self._predict_single(row) for _, row in X.iterrows()])

    def _predict_single(self, row):
        votes = []
        for idx, rule in self.rules_.items():
            if rule.is_realizable(row):
                votes.append(rule.decision)
        if not votes:
            return self.default_class_
        counter = Counter(votes)
        return counter.most_common(1)[0][0]


class GlobalOptClassifier(BaseEstimator, ClassifierMixin):
    def __init__(self, opt_type='length', decision_col='d'):
        self.opt_type = opt_type
        self.decision_col = decision_col
        self.rules_ = {}
        self.default_class_ = None

    def fit(self, X, y):
        T = X.copy()
        T[self.decision_col] = y.values
        self.default_class_ = y.mode()[0]

        rules_rm = induce_rules_for_table(T, self.decision_col, 'RM')
        rules_poly = induce_rules_for_table(T, self.decision_col, 'Poly')

        if self.opt_type == 'length':
            self.rules_ = global_optimization_length(rules_rm, rules_poly)
        else:
            self.rules_ = global_optimization_support(
                rules_rm, rules_poly, T, self.decision_col)
        return self

    def predict(self, X):
        return np.array([self._predict_single(row) for _, row in X.iterrows()])

    def _predict_single(self, row):
        votes = []
        for idx, rule_list in self.rules_.items():
            for rule in rule_list:
                if rule.is_realizable(row):
                    votes.append(rule.decision)
        if not votes:
            return self.default_class_
        counter = Counter(votes)
        return counter.most_common(1)[0][0]

In [9]:
class RuleBasedClassifier(BaseEstimator, ClassifierMixin):
    def __init__(self, heuristic='RM', decision_col='d'):
        self.heuristic = heuristic
        self.decision_col = decision_col
        self.rules_ = {}
        self.default_class_ = None

    def fit(self, X, y):
        if not isinstance(X, pd.DataFrame):
            X = pd.DataFrame(X, columns=self.feature_names_in_)
        else:
            self.feature_names_in_ = list(X.columns)
        T = X.copy()
        y_arr = np.asarray(y)
        T[self.decision_col] = y_arr
        self.default_class_ = pd.Series(y_arr).mode()[0]
        self.rules_ = induce_rules_for_table(T, self.decision_col, self.heuristic)
        return self

    def predict(self, X):
        if not isinstance(X, pd.DataFrame):
            X = pd.DataFrame(X, columns=self.feature_names_in_)
        return np.array([self._predict_single(row) for _, row in X.iterrows()])

    def _predict_single(self, row):
        votes = []
        for idx, rule in self.rules_.items():
            if rule.is_realizable(row):
                votes.append(rule.decision)
        if not votes:
            return self.default_class_
        counter = Counter(votes)
        return counter.most_common(1)[0][0]


class GlobalOptClassifier(BaseEstimator, ClassifierMixin):
    def __init__(self, opt_type='length', decision_col='d'):
        self.opt_type = opt_type
        self.decision_col = decision_col
        self.rules_ = {}
        self.default_class_ = None

    def fit(self, X, y):
        if not isinstance(X, pd.DataFrame):
            X = pd.DataFrame(X, columns=self.feature_names_in_)
        else:
            self.feature_names_in_ = list(X.columns)
        T = X.copy()
        y_arr = np.asarray(y)
        T[self.decision_col] = y_arr
        self.default_class_ = pd.Series(y_arr).mode()[0]
        rules_rm = induce_rules_for_table(T, self.decision_col, 'RM')
        rules_poly = induce_rules_for_table(T, self.decision_col, 'Poly')
        if self.opt_type == 'length':
            self.rules_ = global_optimization_length(rules_rm, rules_poly)
        else:
            self.rules_ = global_optimization_support(
                rules_rm, rules_poly, T, self.decision_col)
        return self

    def predict(self, X):
        if not isinstance(X, pd.DataFrame):
            X = pd.DataFrame(X, columns=self.feature_names_in_)
        return np.array([self._predict_single(row) for _, row in X.iterrows()])

    def _predict_single(self, row):
        votes = []
        for idx, rule_list in self.rules_.items():
            for rule in rule_list:
                if rule.is_realizable(row):
                    votes.append(rule.decision)
        if not votes:
            return self.default_class_
        counter = Counter(votes)
        return counter.most_common(1)[0][0]


In [10]:
def evaluate_classifiers(T, decision_col, cv=10):
    X = T.drop(columns=[decision_col])
    y = T[decision_col]

    min_class_count = y.value_counts().min()
    actual_cv = min(cv, min_class_count)
    if actual_cv < 2:
        actual_cv = 2

    cv_strategy = StratifiedKFold(n_splits=actual_cv, shuffle=True, random_state=42)

    models = {
        'Global-length': GlobalOptClassifier(opt_type='length', decision_col=decision_col),
        'Global-support': GlobalOptClassifier(opt_type='support', decision_col=decision_col),
        'RM': RuleBasedClassifier(heuristic='RM', decision_col=decision_col),
        'Poly': RuleBasedClassifier(heuristic='Poly', decision_col=decision_col)
    }

    results = []
    for name, model in models.items():
        scores = cross_val_score(model, X, y, cv=cv_strategy, scoring='accuracy')
        results.append({
            'Model': name,
            'Accuracy': round(np.mean(scores), 3),
            'Std': round(np.std(scores), 2),
        })

    return pd.DataFrame(results)

In [11]:
def full_analysis(T, decision_col, dataset_name="Dataset"):
    feature_cols = [c for c in T.columns if c != decision_col]
    n_attrs = len(feature_cols)
    n_rows = len(T)

    rules_rm = induce_rules_for_table(T, decision_col, 'RM')
    rules_poly = induce_rules_for_table(T, decision_col, 'Poly')

    global_len = global_optimization_length(rules_rm, rules_poly)
    global_supp = global_optimization_support(rules_rm, rules_poly, T, decision_col)

    stats = {
        'Global-length': compute_statistics(global_len, T, decision_col, is_global=True),
        'Global-support': compute_statistics(global_supp, T, decision_col, is_global=True),
        'RM': compute_statistics(rules_rm, T, decision_col),
        'Poly': compute_statistics(rules_poly, T, decision_col),
    }
    stats_df = build_statistics_dataframe(stats)
    stats_df.insert(0, 'Dataset', dataset_name)

    clf_results = evaluate_classifiers(T, decision_col, cv=10)
    clf_results.insert(0, 'Dataset', dataset_name)

    summary = pd.DataFrame([{
        'Dataset': dataset_name,
        'Atrybuty': n_attrs,
        'Wiersze': n_rows,
        'RM': stats['RM'].get('num_unique_rules', 0),
        'Poly': stats['Poly'].get('num_unique_rules', 0),
        'Globalna optymalizacja względem długości': stats['Global-length'].get('num_unique_rules', 0),
        'Globalna optymalizacja względem wsparcia': stats['Global-support'].get('num_unique_rules', 0),
    }])

    metrics_lenght_df = pd.DataFrame([
        {
            'Dataset': dataset_name,
            'Atrybut': n_attrs,
            'Min RM': stats['RM'].get('min_length'),
            'Avg RM': stats['RM'].get('avg_length'),
            'Max RM': stats['RM'].get('max_length'),
            'Min Poly': stats['Poly'].get('min_length'),
            'Avg Poly': stats['Poly'].get('avg_length'),
            'Max Poly': stats['Poly'].get('max_length'),
        }
    ])
    metrics_supp_df= pd.DataFrame([
        {
            'Dataset': dataset_name,
            'Atrybut': n_attrs,
            'Min RM': stats['RM'].get('min_support'),
            'Avg RM': stats['RM'].get('avg_support'),
            'Max RM': stats['RM'].get('max_support'),
            'Min Poly': stats['Poly'].get('min_support'),
            'Avg Poly': stats['Poly'].get('avg_support'),
            'Max Poly': stats['Poly'].get('max_support'),
        }
    ])
    metrics_opt_sup_df= pd.DataFrame([
        {
            'Dataset': dataset_name,
            'Atrybut': n_attrs,
            'Min Global lenght': stats['Global-length'].get('min_length'),
            'Avg Global-length': stats['Global-length'].get('avg_length'),
            'Max Global-length': stats['Global-length'].get('max_length'),
            'Min Global-support': stats['Global-support'].get('min_support'),
            'Avg Global-support': stats['Global-support'].get('avg_support'),
            'Max Global-support': stats['Global-support'].get('max_support'),
        }
    ])
    metrics_opt_len_df= pd.DataFrame([
        {
            'Dataset': dataset_name,
            'Atrybut': n_attrs,
            'Min Global lenght': stats['Global-length'].get('min_length'),
            'Avg Global-length': stats['Global-length'].get('avg_length'),
            'Max Global-length': stats['Global-length'].get('max_length'),
            'Min Global-support': stats['Global-support'].get('min_length'),
            'Avg Global-support': stats['Global-support'].get('avg_length'),
            'Max Global-support': stats['Global-support'].get('max_length'),
        }
    ])
    
    return {
        'summary': summary,
        'metrics_lenght': metrics_lenght_df,
        'metrics_supp_df': metrics_supp_df,
        'metrics_opt_sup_df':metrics_opt_sup_df,
        'metrics_opt_len_df':metrics_opt_len_df,
        'classification': clf_results,
    }
    

In [12]:
#--- Przyklad z wieloma datasetami ---
datasets = {
    'breast-cancer': ('modified_breast-cancer.csv', 'class'),
    'cars':          ('modified_cars.csv', 'class'),
    'house-votes':   ('modified_house-votes.csv', 'class-name'),  
}

all_results = {}
for name, (path, target) in datasets.items():
    df = pd.read_csv(path)
    all_results[name] = full_analysis(df, target, dataset_name=name)



In [13]:
all_summary         = pd.concat([r['summary']            for r in all_results.values()], ignore_index=True)
all_metrics_length  = pd.concat([r['metrics_lenght']     for r in all_results.values()], ignore_index=True)
all_metrics_supp    = pd.concat([r['metrics_supp_df']    for r in all_results.values()], ignore_index=True)
all_metrics_opt_len = pd.concat([r['metrics_opt_len_df'] for r in all_results.values()], ignore_index=True)
all_metrics_opt_sup = pd.concat([r['metrics_opt_sup_df'] for r in all_results.values()], ignore_index=True)
all_classification  = pd.concat([r['classification']     for r in all_results.values()], ignore_index=True)